# Movimiento de Rodadura

In [6]:
import random
from IPython.display import HTML, display

uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #dee2e6; padding: 18px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 980px; margin: auto;">
    <h3 style="margin-top:0; color: #212529; font-size: 1.3em;">Dinámica del Carrete</h3>
    
    <!-- Cuadro Teórico Resumen -->
    <div style="background: #e9ecef; padding: 12px 16px; border-radius: 6px; font-size: 0.88em; margin-bottom: 12px; color: #212529; line-height: 1.5;">
        <strong>Diagnóstico del Punto de Contacto (M = 2.0 kg, R = 0.50 m, &mu;<sub>s</sub> = 0.35, &mu;<sub>k</sub> = 0.25):</strong>
        <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 10px; margin-top: 6px;">
            <div style="background: #ffffff; padding: 8px 12px; border-radius: 4px; border-left: 4px solid #0d6efd;">
                <b>1. Umbral Crítico:</b><br>
                <i>F<sub>crítico</sub> = 3&mu;<sub>s</sub>Mg = <b>20.58 N</b></i><br>
                f<sub>s,máx</sub> = 6.86 N.
            </div>
            <div style="background: #ffffff; padding: 8px 12px; border-radius: 4px; border-left: 4px solid #198754;">
                <b>2. Rodadura Pura (F &le; 20.58 N):</b><br>
                <i>v<sub>contacto</sub> = v<sub>CM</sub> - R&omega; = <b>0 m/s</b></i><br>
                Giro horario ↻ acoplado con cicloide pura en suelo.
            </div>
            <div style="background: #ffffff; padding: 8px 12px; border-radius: 4px; border-left: 4px solid #dc3545;">
                <b>3. Deslizamiento (F &gt; 20.58 N):</b><br>
                <i>v<sub>contacto</sub> &lt; 0</i> (el borde patina hacia atrás)<br>
                <b>Chispas y vector v<sub>rel</sub></b> visibles en la lupa ampliada.
            </div>
        </div>
    </div>

    <!-- Barra de Controles y Velocidad de Animación -->
    <div style="display: flex; gap: 12px; margin-bottom: 12px; background: #ffffff; padding: 10px 14px; border-radius: 6px; border: 1px solid #ced4da; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; align-items: center; gap: 8px;">
            <button id="play_UID" style="background-color: #198754; color: white; border: none; padding: 6px 14px; border-radius: 4px; cursor: pointer; font-weight: bold;">▶ Ejecutar</button>
            <button id="reset_UID" style="background-color: #6c757d; color: white; border: none; padding: 6px 12px; border-radius: 4px; cursor: pointer; font-weight: bold;">🔄 Reiniciar</button>
        </div>

        <!-- Control de velocidad -->
        <div style="display: flex; align-items: center; gap: 8px; background: #f8f9fa; padding: 4px 10px; border-radius: 4px; border: 1px solid #dee2e6;">
            <label style="font-size: 0.82em; font-weight: bold; color: #495057;">Velocidad:</label>
            <input type="range" id="speed_slider_UID" min="0.04" max="0.30" step="0.02" value="0.10" style="width: 95px;">
            <span id="speed_val_UID" style="font-size: 0.82em; font-weight: bold; color: #0d6efd; min-width: 45px;">0.10x</span>
        </div>

        <div style="display: flex; gap: 8px;">
            <button id="btn_preset_roll_UID" style="background: #e7f1ff; border: 1px solid #0d6efd; color: #084298; padding: 5px 10px; border-radius: 4px; font-size: 0.82em; cursor: pointer; font-weight: bold;">F = 12 N (Rodadura)</button>
            <button id="btn_preset_slip_UID" style="background: #f8d7da; border: 1px solid #dc3545; color: #842029; padding: 5px 10px; border-radius: 4px; font-size: 0.82em; cursor: pointer; font-weight: bold;">F = 28 N (Desliza)</button>
        </div>

        <div id="badge_regimen_UID" style="font-size: 0.85em; font-weight: bold; padding: 5px 12px; border-radius: 4px; background: #d1e7dd; color: #0f5132; border: 1px solid #badbcc;">
            ✓ Rodadura pura
        </div>
    </div>

    <!-- Sliders de Parámetros -->
    <div style="display: grid; grid-template-columns: 1.5fr 1fr; gap: 12px; background: #ffffff; padding: 12px; border-radius: 6px; border: 1px solid #ced4da; margin-bottom: 12px;">
        <div style="background: #f8f9fa; padding: 8px 12px; border-radius: 4px; border: 1px solid #e9ecef;">
            <div style="display: flex; justify-content: space-between; font-size: 0.82em; font-weight: bold; color: #0d6efd;">
                <span>Fuerza horizontal en la cima (F):</span>
                <span id="f_val_UID">12.0 N</span>
            </div>
            <input type="range" id="f_slider_UID" min="2.0" max="36.0" step="0.5" value="12.0" style="width: 100%; margin-top: 4px;">
            <div style="font-size: 0.72em; color: #6c757d; margin-top: 2px;">
                F<sub>crítico</sub> = <b>20.58 N</b>
            </div>
        </div>

        <div style="background: #f8f9fa; padding: 8px 12px; border-radius: 4px; border: 1px solid #e9ecef;">
            <div style="font-size: 0.76em; color: #6c757d; font-weight: bold;">PARÁMETROS</div>
            <div style="font-size: 0.9em; font-weight: bold; color: #212529; margin-top: 3px;">
                &mu;<sub>s</sub> = <span style="color:#198754;">0.35</span> &nbsp;|&nbsp; &mu;<sub>k</sub> = <span style="color:#dc3545;">0.25</span>
            </div>
            <div style="font-size: 0.72em; color: #6c757d;">M = 2.0 kg &nbsp;|&nbsp; R = 0.50 m &nbsp;|&nbsp; I<sub>CM</sub> = 0.5 M R&sup2;</div>
        </div>
    </div>

    <!-- Canvas Principal -->
    <div style="display: flex; justify-content: center; margin-bottom: 12px;">
        <canvas id="canvas_sim_UID" width="940" height="390" style="background: #ffffff; border: 1px solid #ced4da; border-radius: 6px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>

    <!-- Telemetría Dinámica -->
    <div style="background: #ffffff; border: 1px solid #ced4da; border-radius: 6px; padding: 10px; margin-bottom: 10px;">
        <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 8px; text-align: center;">
            <div style="background: #e7f1ff; padding: 6px; border-radius: 4px; border: 1px solid #b6d4fe;">
                <div style="font-size: 0.68em; color: #084298; font-weight: bold;">FUERZA APLICADA (F)</div>
                <div id="lbl_F_UID" style="font-size: 1em; font-weight: bold; color: #0d6efd; font-family: monospace;">12.00 N</div>
            </div>
            <div style="background: #d1e7dd; padding: 6px; border-radius: 4px; border: 1px solid #badbcc;">
                <div style="font-size: 0.68em; color: #0f5132; font-weight: bold;">FRICCIÓN PISO (f)</div>
                <div id="lbl_f_UID" style="font-size: 1em; font-weight: bold; color: #198754; font-family: monospace;">4.00 N</div>
            </div>
            <div style="background: #fff3cd; padding: 6px; border-radius: 4px; border: 1px solid #ffecb5;">
                <div style="font-size: 0.68em; color: #664d03; font-weight: bold;">ACEL. LINEAL (a_CM)</div>
                <div id="lbl_a_UID" style="font-size: 1em; font-weight: bold; color: #856404; font-family: monospace;">8.00 m/s²</div>
            </div>
            <div style="background: #f3e5f5; padding: 6px; border-radius: 4px; border: 1px solid #e1bee7;">
                <div style="font-size: 0.68em; color: #4a148c; font-weight: bold;">ACEL. ANGULAR (&alpha;)</div>
                <div id="lbl_alpha_UID" style="font-size: 1em; font-weight: bold; color: #6f42c1; font-family: monospace;">16.00 rad/s²</div>
            </div>
            <div style="background: #f8d7da; padding: 6px; border-radius: 4px; border: 1px solid #f5c2c7;">
                <div style="font-size: 0.68em; color: #842029; font-weight: bold;">VUELTAS COMPLETADAS</div>
                <div id="lbl_turns_UID" style="font-size: 1em; font-weight: bold; color: #dc3545; font-family: monospace;">0.00 rev</div>
            </div>
            <div id="box_slip_speed_UID" style="background: #e2f0d9; padding: 6px; border-radius: 4px; border: 1px solid #badbcc;">
                <div style="font-size: 0.68em; color: #276a3c; font-weight: bold;">RAPIDEZ PATINAJE (v_rel)</div>
                <div id="lbl_vslip_UID" style="font-size: 1em; font-weight: bold; color: #276a3c; font-family: monospace;">0.00 m/s</div>
            </div>
        </div>
    </div>

    <!-- Filtros de Visualización -->
    <div style="font-size: 0.85em; padding: 8px 12px; background: #e9ecef; border-radius: 5px; display: flex; flex-wrap: wrap; gap: 15px; justify-content: center; user-select: none;">
        <strong>Capas Gráficas:</strong>
        <label style="cursor: pointer;"><input type="checkbox" id="chk_vectors_UID" checked> <b style="color: #0d6efd;">Vectores de Fuerza</b></label>
        <label style="cursor: pointer;"><input type="checkbox" id="chk_trajectory_UID" checked> <b style="color: #d63384;">Trayectoria Cicloide</b></label>
        <label style="cursor: pointer;"><input type="checkbox" id="chk_zoom_UID" checked> <b style="color: #fd7e14;">Lupa Zoom (Contacto)</b></label>
        <label style="cursor: pointer;"><input type="checkbox" id="chk_dcle_UID" checked> <b style="color: #495057;">Diagrama DCLE</b></label>
    </div>
</div>

<script>
(function() {
    function initSimulation() {
        const canvas = document.getElementById('canvas_sim_UID');
        if (!canvas) { setTimeout(initSimulation, 50); return; }
        const ctx = canvas.getContext('2d');

        const f_slider = document.getElementById('f_slider_UID');
        const f_val = document.getElementById('f_val_UID');
        const speed_slider = document.getElementById('speed_slider_UID');
        const speed_val = document.getElementById('speed_val_UID');

        const btn_play = document.getElementById('play_UID');
        const btn_reset = document.getElementById('reset_UID');
        const btn_preset_roll = document.getElementById('btn_preset_roll_UID');
        const btn_preset_slip = document.getElementById('btn_preset_slip_UID');

        const lbl_F = document.getElementById('lbl_F_UID');
        const lbl_f = document.getElementById('lbl_f_UID');
        const lbl_a = document.getElementById('lbl_a_UID');
        const lbl_alpha = document.getElementById('lbl_alpha_UID');
        const lbl_turns = document.getElementById('lbl_turns_UID');
        const lbl_vslip = document.getElementById('lbl_vslip_UID');
        const box_slip_speed = document.getElementById('box_slip_speed_UID');
        const badge_regimen = document.getElementById('badge_regimen_UID');

        const chk_vectors = document.getElementById('chk_vectors_UID');
        const chk_trajectory = document.getElementById('chk_trajectory_UID');
        const chk_zoom = document.getElementById('chk_zoom_UID');
        const chk_dcle = document.getElementById('chk_dcle_UID');

        // Constantes físicas
        const g = 9.8;
        const M = 2.0;       // kg
        const R = 0.50;      // m
        const mu_s = 0.35;
        const mu_k = 0.25;
        const N_norm = M * g;
        const F_crit = 3.0 * mu_s * N_norm; // 20.58 N

        // Parámetros de renderizado
        const floorY = 320;
        const R_px = 34; // px
        const pxPerMeter = R_px / R; // 68 px/m

        // Variables dinámicas
        let isPlaying = true;
        let x_m = 0.0;
        let v_cm = 0;
        let theta = 0;
        let omega = 0;
        let lastTime = performance.now();
        let pathTrace = [];
        let particles = [];

        function drawArrow(context, fx, fy, tx, ty, color, lineWidth=2.2, dashed=false) {
            const dx = tx - fx;
            const dy = ty - fy;
            const len = Math.hypot(dx, dy);
            if (len < 1) return;
            const headlen = Math.min(9, Math.max(5, len * 0.22));
            const angle = Math.atan2(dy, dx);

            context.beginPath();
            context.strokeStyle = color;
            context.lineWidth = lineWidth;
            if(dashed) context.setLineDash([4, 3]);
            context.moveTo(fx, fy);
            context.lineTo(tx, ty);
            context.stroke();
            context.setLineDash([]);

            context.beginPath();
            context.fillStyle = color;
            context.moveTo(tx, ty);
            context.lineTo(tx - headlen * Math.cos(angle - Math.PI / 6), ty - headlen * Math.sin(angle - Math.PI / 6));
            context.lineTo(tx - headlen * Math.cos(angle + Math.PI / 6), ty - headlen * Math.sin(angle + Math.PI / 6));
            context.fill();
        }

        function computePhysics() {
            const F = parseFloat(f_slider.value);
            f_val.innerText = F.toFixed(1) + " N";

            const isRolling = (F <= F_crit);
            let f_friction, a_cm, alpha;

            if (isRolling) {
                f_friction = F / 3.0;
                a_cm = (4.0 * F) / (3.0 * M);
                alpha = a_cm / R;
            } else {
                f_friction = mu_k * N_norm;
                a_cm = (F + f_friction) / M;
                alpha = (2.0 * (F - f_friction)) / (M * R);
            }

            const turns = theta / (2 * Math.PI);
            const v_slip = Math.abs(omega * R - v_cm);

            lbl_F.innerText = F.toFixed(2) + " N";
            lbl_f.innerText = f_friction.toFixed(2) + (isRolling ? " N (fs)" : " N (fk)");
            lbl_a.innerText = a_cm.toFixed(2) + " m/s²";
            lbl_alpha.innerText = alpha.toFixed(2) + " rad/s²";
            lbl_turns.innerText = turns.toFixed(2) + " rev";
            lbl_vslip.innerText = v_slip.toFixed(2) + " m/s";

            if (isRolling) {
                badge_regimen.style.background = "#d1e7dd";
                badge_regimen.style.color = "#0f5132";
                badge_regimen.style.borderColor = "#badbcc";
                badge_regimen.innerText = "✓ Rodadura pura: v_contacto = 0 (Sin resbalamiento)";

                box_slip_speed.style.background = "#e2f0d9";
                box_slip_speed.style.borderColor = "#badbcc";
            } else {
                badge_regimen.style.background = "#f8d7da";
                badge_regimen.style.color = "#842029";
                badge_regimen.style.borderColor = "#f5c2c7";
                badge_regimen.innerText = "⚠ DESLIZAMIENTO: R·α > a_CM (El borde patina hacia atrás)";

                box_slip_speed.style.background = "#f8d7da";
                box_slip_speed.style.borderColor = "#f5c2c7";
            }

            return { F, f_friction, isRolling, a_cm, alpha, v_slip };
        }

        function resetPosition() {
            x_m = 0.0;
            v_cm = 0;
            theta = 0;
            omega = 0;
            pathTrace = [];
            particles = [];
        }

        function draw() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            const p = computePhysics();
            const splitX = 570;

            // División vertical
            ctx.beginPath();
            ctx.strokeStyle = "#e9ecef";
            ctx.lineWidth = 2;
            ctx.moveTo(splitX, 10);
            ctx.lineTo(splitX, canvas.height - 10);
            ctx.stroke();

            // ==========================================================
            // ESCENA FÍSICA (IZQUIERDA)
            // ==========================================================
            ctx.font = "bold 13px sans-serif";
            ctx.fillStyle = "#495057";
            ctx.fillText("Trayectoria Multi-giro (Giro Horario ↻)", 20, 24);

            // Piso con hachurado
            ctx.beginPath();
            ctx.strokeStyle = "#495057";
            ctx.lineWidth = 3;
            ctx.moveTo(15, floorY);
            ctx.lineTo(splitX - 15, floorY);
            ctx.stroke();

            ctx.strokeStyle = "#adb5bd";
            ctx.lineWidth = 1;
            for (let hx = 20; hx < splitX - 20; hx += 16) {
                ctx.beginPath();
                ctx.moveTo(hx, floorY);
                ctx.lineTo(hx - 8, floorY + 12);
                ctx.stroke();
            }

            // Coordenadas en pantalla
            const startX = 42;
            const cx = startX + x_m * pxPerMeter;
            const cy = floorY - R_px;

            const contactX = cx;
            const contactY = floorY;

            // Chispas animadas si hay patinaje
            if (!p.isRolling && isPlaying) {
                for (let k = 0; k < 2; k++) {
                    particles.push({
                        x: contactX,
                        y: contactY - 1,
                        vx: -(Math.random() * 2.5 + 1.2),
                        vy: -(Math.random() * 2.0 + 0.5),
                        life: 1.0
                    });
                }
            }

            for (let i = particles.length - 1; i >= 0; i--) {
                const part = particles[i];
                part.x += part.vx;
                part.y += part.vy;
                part.life -= 0.04;
                if (part.life <= 0) {
                    particles.splice(i, 1);
                } else {
                    ctx.beginPath();
                    ctx.fillStyle = `rgba(253, 126, 20, ${part.life})`;
                    ctx.arc(part.x, part.y, 2, 0, Math.PI * 2);
                    ctx.fill();
                }
            }

            // Punto en el borde: Parte en el piso (θ=0) y sube por atrás girando horario ↻
            const ptX = cx - R_px * Math.sin(theta);
            const ptY = cy + R_px * Math.cos(theta);

            if (isPlaying) {
                pathTrace.push({ x: ptX, y: ptY, isRolling: p.isRolling });
                if (pathTrace.length > 900) pathTrace.shift();
            }

            // Trayectoria cicloide
            if (chk_trajectory.checked && pathTrace.length > 1) {
                for (let i = 1; i < pathTrace.length; i++) {
                    const pA = pathTrace[i - 1];
                    const pB = pathTrace[i];
                    if (Math.abs(pB.x - pA.x) < 40) {
                        ctx.beginPath();
                        ctx.strokeStyle = pB.isRolling ? "rgba(214, 51, 132, 0.55)" : "rgba(220, 53, 69, 0.65)";
                        ctx.lineWidth = pB.isRolling ? 2.2 : 2.5;
                        ctx.moveTo(pA.x, pA.y);
                        ctx.lineTo(pB.x, pB.y);
                        ctx.stroke();
                    }
                }
            }

            // Carrete
            ctx.beginPath();
            ctx.fillStyle = "#f1f3f5";
            ctx.arc(cx, cy, R_px, 0, Math.PI * 2);
            ctx.fill();
            ctx.strokeStyle = "#0d6efd";
            ctx.lineWidth = 2.5;
            ctx.stroke();

            // Tambor interior
            ctx.beginPath();
            ctx.fillStyle = "#cfe2ff";
            ctx.arc(cx, cy, R_px * 0.45, 0, Math.PI * 2);
            ctx.fill();
            ctx.strokeStyle = "#084298";
            ctx.lineWidth = 1.2;
            ctx.stroke();

            // Rayos en rotación HORARIA sincronizada con el punto testigo
            ctx.strokeStyle = "#6c757d";
            ctx.lineWidth = 1.5;
            for (let a = 0; a < 6; a++) {
                const ang = theta + Math.PI / 2 + (a * Math.PI / 3);
                ctx.beginPath();
                ctx.moveTo(cx, cy);
                ctx.lineTo(cx + R_px * Math.cos(ang), cy + R_px * Math.sin(ang));
                ctx.stroke();
            }

            // CM
            ctx.beginPath();
            ctx.fillStyle = "#212529";
            ctx.arc(cx, cy, 3.5, 0, Math.PI * 2);
            ctx.fill();

            // Punto testigo magenta
            ctx.beginPath();
            ctx.fillStyle = "#d63384";
            ctx.arc(ptX, ptY, 4.5, 0, Math.PI * 2);
            ctx.fill();
            ctx.strokeStyle = "#fff";
            ctx.lineWidth = 1.2;
            ctx.stroke();

            // Cable superior
            const topSpoolX = cx;
            const topSpoolY = cy - R_px;
            ctx.beginPath();
            ctx.strokeStyle = "#fd7e14";
            ctx.lineWidth = 2.5;
            ctx.moveTo(topSpoolX, topSpoolY);
            ctx.lineTo(Math.min(splitX - 25, topSpoolX + 75), topSpoolY);
            ctx.stroke();

            // Vectores
            if (chk_vectors.checked) {
                const fVecScale = 1.8;
                drawArrow(ctx, topSpoolX, topSpoolY, topSpoolX + p.F * fVecScale, topSpoolY, "#0d6efd", 2.5);
                ctx.fillStyle = "#0d6efd";
                ctx.font = "bold 11px sans-serif";
                ctx.fillText("F", topSpoolX + p.F * fVecScale + 5, topSpoolY - 4);

                const fColor = p.isRolling ? "#198754" : "#dc3545";
                drawArrow(ctx, contactX, contactY, contactX + p.f_friction * fVecScale * 1.8, contactY, fColor, 2.5);
                ctx.fillStyle = fColor;
                ctx.fillText(p.isRolling ? "f_s" : "f_k", contactX + p.f_friction * fVecScale * 1.8 + 5, contactY + 14);
            }

            // ==========================================================
            // RECUADRO ZOOM (LUPA DE CONTACTO)
            // ==========================================================
            if (chk_zoom.checked) {
                const zBoxX = 25;
                const zBoxY = 48;
                const zBoxW = 210;
                const zBoxH = 140;

                ctx.save();
                ctx.fillStyle = "#ffffff";
                ctx.strokeStyle = p.isRolling ? "#198754" : "#dc3545";
                ctx.lineWidth = 2.5;
                ctx.fillRect(zBoxX, zBoxY, zBoxW, zBoxH);
                ctx.strokeRect(zBoxX, zBoxY, zBoxW, zBoxH);

                ctx.fillStyle = p.isRolling ? "#0f5132" : "#842029";
                ctx.font = "bold 11px sans-serif";
                ctx.fillText(p.isRolling ? "🔍 ZOOM: Contacto en Reposo" : "🔍 ZOOM: Patina hacia atrás!", zBoxX + 8, zBoxY + 16);

                ctx.beginPath();
                ctx.rect(zBoxX + 2, zBoxY + 22, zBoxW - 4, zBoxH - 24);
                ctx.clip();

                const zoomFactor = 2.6;
                const zFloorY = zBoxY + 88;
                const zCenterX = zBoxX + zBoxW / 2;
                const zR = R_px * zoomFactor;

                ctx.beginPath();
                ctx.strokeStyle = "#495057";
                ctx.lineWidth = 4;
                ctx.moveTo(zBoxX, zFloorY);
                ctx.lineTo(zBoxX + zBoxW, zFloorY);
                ctx.stroke();

                ctx.strokeStyle = "#6c757d";
                ctx.lineWidth = 1.5;
                const groundOffset = -(x_m * pxPerMeter * zoomFactor) % 25;
                for (let gx = zBoxX - 30; gx < zBoxX + zBoxW + 30; gx += 25) {
                    const posX = gx + groundOffset;
                    ctx.beginPath();
                    ctx.moveTo(posX, zFloorY);
                    ctx.lineTo(posX - 6, zFloorY + 12);
                    ctx.stroke();
                }

                ctx.beginPath();
                ctx.fillStyle = "#f8f9fa";
                ctx.arc(zCenterX, zFloorY - zR, zR, 0, Math.PI * 2);
                ctx.fill();
                ctx.strokeStyle = "#0d6efd";
                ctx.lineWidth = 3;
                ctx.stroke();

                // Rayos ampliados con rotación horaria
                ctx.strokeStyle = "#6c757d";
                ctx.lineWidth = 2.2;
                for (let a = 0; a < 8; a++) {
                    const ang = theta + Math.PI / 2 + (a * Math.PI / 4);
                    ctx.beginPath();
                    ctx.moveTo(zCenterX, zFloorY - zR);
                    ctx.lineTo(zCenterX + zR * Math.cos(ang), (zFloorY - zR) + zR * Math.sin(ang));
                    ctx.stroke();
                }

                // Punto testigo ampliado
                const zPtX = zCenterX - zR * Math.sin(theta);
                const zPtY = (zFloorY - zR) + zR * Math.cos(theta);
                ctx.beginPath();
                ctx.fillStyle = "#d63384";
                ctx.arc(zPtX, zPtY, 6, 0, Math.PI * 2);
                ctx.fill();
                ctx.strokeStyle = "#fff";
                ctx.lineWidth = 1.5;
                ctx.stroke();

                // Vector velocidad relativa
                const v_rel = v_cm - (omega * R);
                if (Math.abs(v_rel) > 0.05) {
                    const vArrowLen = Math.max(-75, Math.min(75, v_rel * 22));
                    drawArrow(ctx, zCenterX, zFloorY - 6, zCenterX + vArrowLen, zFloorY - 6, "#dc3545", 3);
                    ctx.fillStyle = "#dc3545";
                    ctx.font = "bold 11px sans-serif";
                    ctx.fillText("v_rel = " + Math.abs(v_rel).toFixed(2) + " m/s", zCenterX - 65, zFloorY - 14);
                } else {
                    ctx.fillStyle = "#198754";
                    ctx.font = "bold 11px sans-serif";
                    ctx.fillText("v_rel = 0.00 m/s (Adherido)", zCenterX - 65, zFloorY - 12);
                }

                ctx.restore();

                ctx.strokeStyle = "rgba(253, 126, 20, 0.35)";
                ctx.setLineDash([2, 3]);
                ctx.lineWidth = 1.2;
                ctx.beginPath();
                ctx.moveTo(zBoxX + zBoxW, zBoxY + zBoxH / 2);
                ctx.lineTo(contactX, contactY);
                ctx.stroke();
                ctx.setLineDash([]);
            }

            // ==========================================================
            // DIAGRAMA DCLE (DERECHA)
            // ==========================================================
            const dclX = 760;
            const dclY = 195;
            const dclR = 80;

            ctx.fillStyle = "#212529";
            ctx.font = "bold 14px sans-serif";
            ctx.fillText("Diagrama de Fuerzas (DCLE)", 610, 24);

            if (chk_dcle.checked) {
                ctx.beginPath();
                ctx.fillStyle = "#f8f9fa";
                ctx.arc(dclX, dclY, dclR, 0, Math.PI * 2);
                ctx.fill();
                ctx.strokeStyle = "#adb5bd";
                ctx.lineWidth = 2;
                ctx.stroke();

                ctx.strokeStyle = "#ced4da";
                ctx.setLineDash([3, 3]);
                ctx.beginPath();
                ctx.moveTo(dclX - dclR - 20, dclY); ctx.lineTo(dclX + dclR + 30, dclY);
                ctx.moveTo(dclX, dclY - dclR - 25); ctx.lineTo(dclX, dclY + dclR + 30);
                ctx.stroke();
                ctx.setLineDash([]);

                const vecScale = 2.0;

                drawArrow(ctx, dclX, dclY - dclR, dclX + p.F * vecScale, dclY - dclR, "#0d6efd", 3.2);
                ctx.fillStyle = "#0d6efd";
                ctx.font = "bold 12px sans-serif";
                ctx.fillText("F", dclX + p.F * vecScale + 6, dclY - dclR - 4);

                const fCol = p.isRolling ? "#198754" : "#dc3545";
                drawArrow(ctx, dclX, dclY + dclR, dclX + p.f_friction * vecScale * 1.8, dclY + dclR, fCol, 3.2);
                ctx.fillStyle = fCol;
                ctx.fillText(p.isRolling ? "f_s" : "f_k", dclX + p.f_friction * vecScale * 1.8 + 6, dclY + dclR + 15);

                drawArrow(ctx, dclX, dclY + dclR, dclX, dclY + dclR - 50, "#495057", 2.5);
                ctx.fillStyle = "#495057";
                ctx.fillText("N", dclX + 8, dclY + dclR - 40);

                drawArrow(ctx, dclX, dclY, dclX, dclY + 50, "#dc3545", 2.5);
                ctx.fillStyle = "#dc3545";
                ctx.fillText("P = M·g", dclX + 8, dclY + 45);

                ctx.beginPath();
                ctx.fillStyle = "#212529";
                ctx.arc(dclX, dclY, 4, 0, Math.PI * 2);
                ctx.fill();

                ctx.fillStyle = "#212529";
                ctx.font = "bold 12px monospace";
                ctx.fillText("∑ F_x = F + f = M·a_CM", 620, dclY + 115);
                ctx.fillText("∑ τ_CM = F·R - f·R = I_CM·α", 620, dclY + 135);
                ctx.fillStyle = p.isRolling ? "#198754" : "#dc3545";
                ctx.fillText(p.isRolling ? "Condición: a_CM = R·α" : "Patinaje: R·α > a_CM", 620, dclY + 155);
            }
        }

        function animate(timestamp) {
            let dt = (timestamp - lastTime) / 1000;
            if (dt > 0.08) dt = 0.08;
            lastTime = timestamp;

            const timeScale = parseFloat(speed_slider.value);
            speed_val.innerText = timeScale.toFixed(2) + "x";

            if (isPlaying) {
                const p = computePhysics();
                const sim_dt = dt * timeScale;

                v_cm += p.a_cm * sim_dt;
                x_m += v_cm * sim_dt;

                omega += p.alpha * sim_dt;
                theta += omega * sim_dt;

                const max_x_px = 510;
                if ((42 + x_m * pxPerMeter) > max_x_px) {
                    resetPosition();
                }
            }

            draw();
            requestAnimationFrame(animate);
        }

        // Listeners
        f_slider.addEventListener('input', () => { computePhysics(); draw(); });

        btn_preset_roll.addEventListener('click', () => {
            f_slider.value = 12.0;
            resetPosition();
            computePhysics();
            draw();
        });

        btn_preset_slip.addEventListener('click', () => {
            f_slider.value = 28.0;
            resetPosition();
            computePhysics();
            draw();
        });

        [chk_vectors, chk_trajectory, chk_zoom, chk_dcle].forEach(c => {
            c.addEventListener('change', draw);
        });

        btn_play.addEventListener('click', () => {
            isPlaying = !isPlaying;
            btn_play.innerText = isPlaying ? "⏸ Pausar" : "▶ Ejecutar";
            btn_play.style.backgroundColor = isPlaying ? "#ffc107" : "#198754";
            btn_play.style.color = isPlaying ? "#000" : "#fff";
        });

        btn_reset.addEventListener('click', () => {
            resetPosition();
            draw();
        });

        computePhysics();
        requestAnimationFrame(animate);
    }

    initSimulation();
})();
</script>
"""

display(HTML(simulacion_html.replace("UID", uid)))